## Рекомендация тарифов

#### В вашем распоряжении данные о поведении клиентов, которые уже перешли на эти тарифы (из предыдущего проекта «Статистический анализ данных»). Нужно построить модель для задачи классификации, которая выберет подходящий тариф. Предобработка данных не понадобится — мы её уже сделали.

#### Построем модель с максимально большим значением *accuracy*. Чтобы сдать проект успешно, нужно довести долю правильных ответов по крайней мере до 0.75. Проверим *accuracy* на тестовой выборке.

## Откроем и изучим файл

In [53]:
import pandas as pd
from sklearn.model_selection import train_test_split

pd.options.display.max_columns = None
pd.set_option('display.float_format', '{:.2f}'.format)

df=pd.read_csv('users_behavior.csv')

display(df,df.info(),df.describe())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


,calls,minutes,messages,mb_used,is_ultra
0,40.00,311.90,83.00,19915.42,0
1,85.00,516.75,56.00,22696.96,0
2,77.00,467.66,86.00,21060.45,0
3,106.00,745.53,81.00,8437.39,1
4,66.00,418.74,1.00,14502.75,0
...,...,...,...,...,...
3209,122.00,910.98,20.00,35124.90,1
3210,25.00,190.36,0.00,3275.61,0
3211,97.00,634.44,70.00,13974.06,0
3212,64.00,462.32,90.00,31239.78,0


None

,calls,minutes,messages,mb_used,is_ultra
count,3214.00,3214.00,3214.00,3214.00,3214.00
mean,63.04,438.21,38.28,17207.67,0.31
std,33.24,234.57,36.15,7570.97,0.46
min,0.00,0.00,0.00,0.00,0.00
25%,40.00,274.58,9.00,12491.90,0.00
50%,62.00,430.60,30.00,16943.24,0.00
75%,82.00,571.93,57.00,21424.70,1.00
max,244.00,1632.06,224.00,49745.73,1.00


#### В датасет содержится 3214 наблюдений, каждое наблюдение имеет признаки calls, minutes, messages, mb_used (кол-во звонков, потраченные минуты, кол-во сообщений, использование трафика) и целевой признак is_ultra (0 - тариф смарт, 1 - тариф ультра)

## Разобъем данные на выборки

In [54]:
df_train,df_test=train_test_split(df,test_size=0.25,random_state=2281337)

df_train,df_val=train_test_split(df_train,test_size=0.25,random_state=2281337)


features_train,target_train=df_train.drop(columns='is_ultra'),df_train['is_ultra']


features_val,target_val=df_val.drop(columns='is_ultra'),df_val['is_ultra']


features_test,target_test=df_test.drop(columns='is_ultra'),df_test['is_ultra']

display(len(features_train),len(features_val),len(features_test))

1807

603

804

#### Данные разбил в отношении 56.25/18.75/25 (обучение, валидация, тест). При делении 60/20/20 и 50/25/25 результаты были хуже

## Исследуем модели

#### Модель решающего дерева:
#### Выберем наилучшее значение глубины дерева, а также поэксперементируем с criterion    

In [55]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

best_model_tree=None
best_score_tree=0

for i in range(1,10):
    model=DecisionTreeClassifier(random_state=12345,max_depth=i,criterion='gini')

    model.fit(features_train,target_train)

    answers=model.predict(features_val)
 
    score=(accuracy_score(answers,target_val))
    
    if score>best_score_tree:
        best_model_tree=model
        best_score_tree=score

display(best_score_tree,best_model_tree.get_params())

0.814262023217247

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': 6,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': 12345,
 'splitter': 'best'}

#### Лучшее значение accuracy = 0.8143 достигается при глубине 6. При этом изменение criterion существенно не повлияло на точность модели. 

#### Случайный лес:

#### Выберем наилучшее значение глубины дерева и кол-во деревьев, а также поэксперементируем с criterion    

In [56]:
from sklearn.ensemble import RandomForestClassifier

best_model_rand_for=None
best_score_rand_for=0

for i in range(10,51,10):
    for j in range(1,10):
        model=RandomForestClassifier(random_state=12345,n_estimators=i,max_depth=j,criterion='gini')

        model.fit(features_train,target_train)

        answers=model.predict(features_val)
    
        score=(accuracy_score(answers,target_val))
        
        if score>best_score_rand_for:
            best_model_rand_for=model
            best_score_rand_for=score

display(best_score_rand_for,best_model_rand_for.get_params())

#0.8308457711442786

0.8325041459369817

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': 8,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 20,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 12345,
 'verbose': 0,
 'warm_start': False}

#### Наилучшие параметры случайного леса: 
* максимальная глубина = 8, 
* кол-во деревьев = 20,  
* criterion = 'gini'. Стоит отметить что изменение criterion влекло незначительное изменение точности  (при 'gini' - 0.8325, при 'entropy' - 0.8308).
* При таких параметрах точность accuracy = 0.8325

#### Логистическая регрессия

In [57]:
from sklearn.linear_model import LogisticRegression


model_log=LogisticRegression()

model_log.fit(features_train,target_train)

answers=model_log.predict(features_val)
    
score=(accuracy_score(answers,target_val))


display(score,model_log.get_params())

0.7645107794361525

{'C': 1.0,
 'class_weight': None,
 'dual': False,
 'fit_intercept': True,
 'intercept_scaling': 1,
 'l1_ratio': None,
 'max_iter': 100,
 'multi_class': 'deprecated',
 'n_jobs': None,
 'penalty': 'l2',
 'random_state': None,
 'solver': 'lbfgs',
 'tol': 0.0001,
 'verbose': 0,
 'warm_start': False}

#### Точность accuracy на логистической регрессии = 0.7645

## Проверим модели на тестовой выборке

#### Решающее дерево:

In [58]:
answers_tree=best_model_tree.predict(features_test)

display(accuracy_score(answers_tree,target_test))

0.804726368159204

#### Случайный лес

In [59]:
answers_rand_for=best_model_rand_for.predict(features_test)

display(accuracy_score(answers_rand_for,target_test))

0.8208955223880597

#### Логистическая регрессия

In [60]:
answers_log=model_log.predict(features_test)

display(accuracy_score(answers_log,target_test))

0.7549751243781094

#### Наилучшая точность на тестовой выборке у рандомного леса, точность accuray = 0.8209

## Проверим модели на адекватность

#### Дерево классификатор

In [61]:
from sklearn.metrics import precision_score,recall_score,f1_score

display('Precision:',precision_score(target_test,answers_tree))

display('Recall:',recall_score(target_test,answers_tree))

display('f1:',f1_score(target_test,answers_tree))

'Precision:'

0.7835820895522388

'Recall:'

0.45064377682403434

'f1:'

0.5722070844686649

#### Случайный лес

In [62]:
display('Precision:',precision_score(target_test,answers_rand_for))

display('Recall:',recall_score(target_test,answers_rand_for))

display('f1:',f1_score(target_test,answers_rand_for))

'Precision:'

0.8156028368794326

'Recall:'

0.49356223175965663

'f1:'

0.6149732620320856

#### Логистическая регрессия

In [63]:
display('Precision:',precision_score(target_test,answers_log))

display('Recall:',recall_score(target_test,answers_log))

display('f1:',f1_score(target_test,answers_log))

'Precision:'

0.7727272727272727

'Recall:'

0.21888412017167383

'f1:'

0.3411371237458194


#### Лучшие показатели preciption recall и f1 все также у рандомного леса:
* racall = 0.4936 (показывает какую долю объектов класса 1 модель смогла найти.) - модель находит только 49% всех пользователей Ultra.
* precision = 0.8156 (показывает долю объектов, которые модель назвала классом 1 и которые действительно являются классом 1) - среди объектов, которые модель назвала тарифом Ultra, 81.5% действительно являются Ultra.
* f1 = 0.615 (среднее гармоническое из precision и recall)


## Вывод

#### В ходе работы была решена задача классификации — построена модель, рекомендующая пользователю подходящий тариф на основе его поведения. В качестве признаков использовались: количество звонков (calls), количество потраченных минут (minutes), число сообщений (messages) и объём интернет-трафика (mb_used). Целевым признаком являлась переменная is_ultra, показывающая, использует ли клиент тариф Ultra (1) или Smart (0).

#### Данные были разделены на обучающую, валидационную и тестовую выборки в пропорции 50/25/25. На обучающей выборке были обучены несколько моделей машинного обучения: решающее дерево, случайный лес и логистическая регрессия. Для моделей решающего дерева и случайного леса был выполнен подбор гиперпараметров (глубина дерева и количество деревьев).

#### Лучшие результаты на валидационной выборке показала модель случайного леса с параметрами:

* максимальная глубина дерева — 8

* количество деревьев — 20

* Точность модели на валидационной выборке составила 0.8325.

#### Далее модели были проверены на тестовой выборке. Полученные значения accuracy:

* Decision Tree — 0.8047

* Random Forest — 0.8209

* Logistic Regression — 0.7550

#### Наилучший результат снова показал случайный лес, достигнув точности 0.8209, что превышает требуемый порог 0.75.

#### Дополнительно были рассчитаны метрики precision, recall и F1-мера. Для модели случайного леса значения составили:

* Precision = 0.8156

* Recall = 0.4936

* F1-мера = 0.615

#### Это означает, что модель достаточно точно определяет пользователей тарифа Ultra, однако обнаруживает не все такие случаи.

#### Таким образом, для задачи рекомендации тарифов наиболее подходящей оказалась модель Random Forest, которая показала наилучшее качество классификации среди рассмотренных моделей и удовлетворяет требованиям задачи.